<a href="https://colab.research.google.com/github/ramanchauhan2271-dev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Note: due to sample size constraints, this development slice covers
2026-03-01 to 2026-03-02 within the March 2026 window.

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import pandas as pd

login(token=userdata.get('HF_TOKEN'))

from datasets import load_dataset

# streaming=True -> no full download, reads on the fly
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True
)

# Take only rows from March 2026, and stop after 50,000 rows
def is_march_2026(row):
    return row["report_date"].year == 2026 and row["report_date"].month == 3

filtered_stream = dataset["train"].filter(is_march_2026)

# Pull only first 50,000 matching rows into a list
rows = []
for i, row in enumerate(filtered_stream):
    rows.append(row)
    if len(rows) >= 50000:
        break

df = pd.DataFrame(rows)
print("Rows in df:", len(df))
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
# Verify grain: one row = one (content_hash_id, report_date) pair
df.groupby(["content_hash_id", "report_date"]).size().max()
# should print 1 if grain holds

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_impressions, gsc_clicks, gsc_avg_position (prior days) —
  knowable at decision time.
Label: gsc_clicks (next period) or ga4_sessions (next period) — the
  outcome we're trying to predict/rank, only knowable after the fact.
Context: content_hash_id, client_hash_id, report_date — identifiers,
  not fed to the model directly.
Excluded: ai_chatgpt, ai_perplexity, ai_gemini (AI-referral session
  columns) — deliberately excluded for this lane since our focus is
  refresh/content scoring based on organic search performance, not
  AI-traffic attribution.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Query 1: grain check
query1 = """
SELECT page, month, COUNT(*) as row_count
FROM `flyrank.internship_warehouse.search_console`
WHERE month = '2026-03'
GROUP BY page, month
HAVING COUNT(*) > 1
"""
# should return 0 rows if grain is really (page, month)

# Query 2: row count + date span for your slice
query2 = """
SELECT COUNT(*) as total_rows, MIN(month) as start, MAX(month) as end
FROM `flyrank.internship_warehouse.search_console`
WHERE month = '2026-03'
"""

# Query 3: availability check
query3 = """
SELECT COUNT(*) as available_rows
FROM `flyrank.internship_warehouse.search_console`
WHERE month = '2026-03' AND clicks IS NOT NULL AND ctr IS NOT NULL IS TRUE
"""

In [ ]:
# Query 2: row count + date span
print("Total rows:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

In [ ]:
# Query 3: availability check
available = df[df["gsc_data_available"] == True]
print("Available rows:", len(available), "out of", len(df))

5. Five features + the trap

Five features max, each with an "available at decision moment because…" line.
Then: add one label-derived column on purpose, watch the score jump, delete it,
report the honest number.


In [ ]:
# Feature frame for the Refresh / Content Opportunity Scoring lane
# Each feature: knowable at the decision moment because...

features_df = df[[
    "content_hash_id", "report_date",
    "gsc_impressions",       # available: closed metric, logged per day
    "gsc_avg_position",       # available: closed metric
    "gsc_data_available"      # available: static flag, known at load time
]].copy()

features_df.head()

In [ ]:
# THE TRAP: add a label-derived column on purpose
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = features_df.drop(columns=["content_hash_id", "report_date"]).fillna(0)
y = (df["gsc_clicks"] > 0).astype(int)  # the label: did this content get any clicks

# --- WITHOUT leakage ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC (no leakage):", honest_score)

# --- WITH leakage (the trap) ---
X_leaky = X.copy()
X_leaky["leaked_feature"] = y  # directly derived from the label!

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_score = roc_auc_score(y_test_l, model_leaky.predict_proba(X_test_l)[:, 1])
print("Leaky AUC (with trap):", leaky_score)

# --- Delete the leaked column, confirm honest score stands ---
X_leaky = X_leaky.drop(columns=["leaked_feature"])
print("\nLeakage removed. Honest AUC score stands at:", honest_score)

The trap: adding a column derived directly from the label (leaked_feature)
pushed AUC from 0.94 to 1.0 — an unrealistic jump. This confirms the leakage
lesson from notebook 02: any feature that encodes the outcome itself will
inflate performance without being usable in real deployment, since it isn't
knowable at decision time. Removing it restores the honest number.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell us: (1) intent behind a search query, only
click behavior; (2) performance before a page's indexing history
began (unbalanced history for newer pages); (3) true causal effect
of a refresh — only correlation between refresh timing and later CTR.

Five features used (see code cell above for full frame):
1. gsc_impressions — available: closed metric, logged per day
2. gsc_avg_position — available: closed metric
3. gsc_data_available — available: static flag, known at load time


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.